# Training and interruption recovery
Default execution is a single smoke epoch on 32 training and 16 validation images, drawn from the development manifest. This verifies computation and checkpointing; its scores are not performance evidence.

In [ ]:
from pathlib import Path
import os
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if not (ROOT / "configs/cpu.yaml").exists():
    raise RuntimeError("Open this notebook from the project root or notebooks directory")
os.chdir(ROOT)
import yaml
from ml.runtime import configure
config = yaml.safe_load(Path("configs/cpu.yaml").read_text())
configure(config["seed"], config["threads"])

In [ ]:
from ml.training.trainer import train
RUN_SMOKE_TEST = False  # Set True to intentionally run the bounded smoke test.
if RUN_SMOKE_TEST:
    history = train(config, smoke=True)
else:
    print("Smoke test is opt-in; no training started.")

## Full training requires approval
Do not enable this cell until the project owner approves full training. Initial configuration is five head-training epochs with early stopping; fine-tuning is disabled. Model selection uses validation ROC-AUC and a fixed 0.5 threshold. Test data is never used here.

In [ ]:
FULL_TRAINING_APPROVED = False
if FULL_TRAINING_APPROVED:
    history = train(config, smoke=False, approved_full_training=True)
else:
    print("Full training awaits explicit approval.")

## Resume a saved run
Interrupt using the Jupyter stop button. The last checkpoint stores a consistent epoch boundary: model, optimizer, scheduler, RNG states, split fingerprint, config, and history. An incomplete epoch is replayed. Use exactly the same configuration. Smoke artifacts cannot be served as trained models.

In [ ]:
RESUME_PATH = None  # e.g. artifacts/checkpoints/smoke/last.pt
if RESUME_PATH:
    from ml.training.checkpoint import read_checkpoint
    saved = read_checkpoint(RESUME_PATH)
    history = train(config, smoke=saved["smoke"],
                    approved_full_training=FULL_TRAINING_APPROVED, resume=RESUME_PATH)

In [ ]:
import json
from IPython.display import display, Image
history_path = Path(config["checkpoint_dir"]) / "smoke/history.json"
if history_path.exists():
    import pandas as pd
    display(pd.json_normalize(json.loads(history_path.read_text())))
    display(Image(filename=str(history_path.parent / "training_curves.png")))